# XGBoost Submission Workflow

This notebook shows how an XGBoost model can log a reconstructable submission bundle to MLflow. The important part is that `weights.parquet` is still just the backtested portfolio allocation, while `model_submission/` contains the trained model file plus the metadata needed to recreate inference later.

## Setup

This example assumes `xgboost` is installed in your environment. If needed, install it with `python3 -m pip install xgboost` before running the notebook.

In [1]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
model_name = 'xgboost_submission_example'
horizon = 5
output_dir = repo_root / 'runs' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Repo root:', repo_root)
print('Dataset:', dataset_name, spec.name)
print('Tickers:', len(spec.tickers))
print('Splits:', splits)

/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
Dataset: shared_set_2 growth_tech_innovation
Tickers: 26
Splits: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## Build Features And Target

In [3]:
prices = load_prices(dataset_name, repo_root=repo_root)

feature_names = [
    'momentum_20d',
    'momentum_60d',
    'vol_20d',
    'vol_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'beta_20d_spy',
    'bollinger_z_20d',
    'excess_return_20d_vs_spy',
]

features = build_features(prices, feature_names=feature_names)
target = make_forward_return_target(prices, horizon=horizon)
target_col = f'forward_return_{horizon}d'

model_frame = (
    features
    .merge(target[['date', 'ticker', target_col]], on=['date', 'ticker'], how='left')
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=feature_names + [target_col])
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

print('Model frame:', model_frame.shape)
display(model_frame.head())

Model frame: (76765, 14)


,date,ticker,momentum_20d,momentum_60d,vol_20d,vol_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,beta_20d_spy,bollinger_z_20d,excess_return_20d_vs_spy,forward_return_5d
0,2014-03-31,AAPL,0.017016,-0.023823,0.006837,0.014637,50.700755,0.006098,0.014113,-1.268528,0.415704,0.684347,0.001580,-0.024723
1,2014-04-01,AAPL,0.019596,0.007232,0.006966,0.014410,54.962816,0.014312,0.023227,-0.625223,0.468963,1.516221,0.011594,-0.033619
2,2014-04-02,AAPL,0.019141,0.003434,0.006963,0.014395,63.014516,0.015029,0.025053,-0.962614,0.465291,1.500516,0.008683,-0.022542
3,2014-04-03,AAPL,0.015148,0.003657,0.007125,0.014393,66.199900,0.007237,0.018312,-1.238125,0.501710,0.722552,0.008333,-0.028416
4,2014-04-04,AAPL,0.002602,-0.015560,0.007726,0.014467,55.243859,-0.005921,0.005940,0.682547,0.631208,-0.596826,0.008112,-0.022959


## Train XGBoost

XGBoost does not need feature standardization for this tabular setup, so the submission manifest records `scaler: none`.

In [4]:
train = slice_split(model_frame, dataset_name, 'train', repo_root=repo_root)
val = slice_split(model_frame, dataset_name, 'val', repo_root=repo_root)
test = slice_split(model_frame, dataset_name, 'test', repo_root=repo_root)

X_train = train[feature_names]
y_train = train[target_col]
X_val = val[feature_names]
y_val = val[target_col]
X_test = test[feature_names]

xgb_params = {
    'n_estimators': 300,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 10,
    'objective': 'reg:squarederror',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

model = xgb.XGBRegressor(**xgb_params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

val_pred = model.predict(X_val)
val_rmse = float(np.sqrt(np.mean((val_pred - y_val.to_numpy(dtype=float)) ** 2)))
print('Validation RMSE:', val_rmse)

model_path = output_dir / 'xgboost_model.json'
model.save_model(model_path)
print('Saved model:', model_path)

Validation RMSE: 0.05638975924948329
Saved model: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/runs/xgboost_submission_example/xgboost_model.json


## Predictions, Portfolio, And Backtest

In [5]:
test_scores = model.predict(X_test)

predictions = test[['date', 'ticker']].copy()
predictions['horizon'] = horizon
predictions['expected_return'] = test_scores

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

portfolio = weights_from_predictions_rank_long_only(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=dataset_name, repo_root=repo_root)

result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
metrics = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

print('Weights:', validated_weights.shape)
print('Metrics:')
for key, value in sorted(metrics.items()):
    print(f'  {key}: {value:.6f}')

Weights: (998, 26)
Metrics:
  annual_return: 0.033486
  annual_volatility: 0.174827
  average_turnover: 0.144086
  benchmark_total_return: 0.507582
  calmar: 0.077107
  excess_return_vs_benchmark: -0.023792
  max_drawdown: -0.434282
  sharpe: 0.191539
  sortino: 0.162945
  total_return: 0.483790


## Log MLflow Run And Model Submission Bundle

The `log_model_submission(...)` call packages the saved XGBoost model with feature order, target, horizon, preprocessing, model config, and this source notebook. That is the part needed to recreate model inference later.

In [6]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=model_name,
    dataset_name=dataset_name,
    tags={
        'workflow': 'xgboost_submission_workflow',
        'model_family': 'xgboost',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        'model_name': model_name,
        'dataset_name': dataset_name,
        'horizon': horizon,
        'feature_count': len(feature_names),
        'feature_list': ','.join(feature_names),
        'portfolio_builder': 'weights_from_predictions_rank_long_only',
        'val_rmse': val_rmse,
        **xgb_params,
    })

    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

    manifest = log_model_submission(
        {'xgboost_model': model_path},
        model_name=model_name,
        model_family='xgboost',
        feature_names=feature_names,
        target=target_col,
        horizon=horizon,
        preprocessing={'scaler': 'none'},
        model_config={
            'library': 'xgboost',
            'artifact_format': 'xgboost_json',
            'params': xgb_params,
            'portfolio_builder': 'weights_from_predictions_rank_long_only',
        },
        source_files=[repo_root / 'notebooks' / 'templates' / 'xgboost_submission_workflow.ipynb'],
        notes='Example XGBoost model submission bundle.',
    )

print('Logged model submission manifest:')
manifest

MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
🏃 View run xgboost_submission_example at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1/runs/2590f1637d834e4ea578aa81379c5c2d
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1
Logged model submission manifest:


{'model_name': 'xgboost_submission_example',
 'model_family': 'xgboost',
 'target': 'forward_return_5d',
 'horizon': 5,
 'feature_names': ['momentum_20d',
  'momentum_60d',
  'vol_20d',
  'vol_60d',
  'rsi_14',
  'price_to_sma_20d',
  'price_to_sma_50d',
  'volume_zscore_20d',
  'beta_20d_spy',
  'bollinger_z_20d',
  'excess_return_20d_vs_spy'],
 'preprocessing': {'scaler': 'none'},
 'model_config': {'library': 'xgboost',
  'artifact_format': 'xgboost_json',
  'params': {'n_estimators': 300,
   'max_depth': 4,
   'learning_rate': 0.05,
   'subsample': 0.8,
   'colsample_bytree': 0.8,
   'min_child_weight': 10,
   'objective': 'reg:squarederror',
   'random_state': 42,
   'n_jobs': -1,
   'verbosity': 0},
  'portfolio_builder': 'weights_from_predictions_rank_long_only'},
 'notes': 'Example XGBoost model submission bundle.',
 'artifact_files': ['artifacts/xgboost_model.json'],
 'artifact_map': {'xgboost_model': 'artifacts/xgboost_model.json'},
 'source_files': ['source/xgboost_submission

## Recreate Inference From The Logged Bundle

Later, the official evaluator can download `model_submission/manifest.json`, load `model_submission/artifacts/xgboost_model.json`, rebuild the same features in the manifest order, and call `xgb.XGBRegressor().load_model(...)` before prediction.